###  1. Defining the schema for the circuit input

- getting the batch folder name as input using dbutils.widgets

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00.Common/01.Environment-config


In [0]:
%run "../00.Common/02.Helper_Notebook_Bronze"

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
from pyspark.sql.types import StructField, StringType, DoubleType, StructType
circuit_schema = StructType([
    StructField('circuitId', StringType()),
    StructField('url', StringType()),
    StructField('circuitname', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType())
])

### 2. Reading the circuits.csv file from the files location

In [0]:
circuits_df = (
    spark.read
    .format('csv')
    .option('header','true')
    .schema(circuit_schema)
    .load(source_file)
     )

### 3. View the circuit df

In [0]:

display(circuits_df)

### 4. Adding the file ingestion timestamp and sourcefile metadata in the df

In [0]:
from pyspark.sql import functions as F
final_circuit = add_timestamp_metadata(circuits_df)

### 5. Writing the final dataframe into the table under bronze schema

In [0]:
write_to_bronze (
input_df=final_circuit,
target_table = table_name,
batch_id = v_batch_id
)

In [0]:
display(spark.table(table_name))